# AudioSep Ablation Study — Kaggle Notebook

This notebook is the **cloud orchestrator** for the Conv-TasNet DSC ablation study.
All core logic lives in `src/`; this notebook installs dependencies, configures paths,
kicks off training/evaluation for each experiment, and archives the results.

## Experiments
| Config | DSC layers | Description |
|--------|------------|-------------|
| `baseline` | 0 | Pure Conv-TasNet (no DSC) |
| `dsc5`     | 5 | First 5 / 24 TCN blocks → DSC |
| `dsc10`    | 10 | First 10 / 24 TCN blocks → DSC |
| `dsc20`    | 20 | First 20 / 24 TCN blocks → DSC |
| `dsc_full` | 24 | All 24 / 24 TCN blocks → DSC (full DSC) |

## Layout on Kaggle
```
/kaggle/
  input/
    musdb18hq/           ← Kaggle dataset (add via Settings → Add Data)
      train/<track>/mixture.wav  ...
      test/<track>/mixture.wav   ...
    audiosepablationstudy/       ← this repo (add as Kaggle dataset)
      src/  configs/  ...
  working/
    checkpoints/         ← saved during the run
    results/             ← CSV metrics
    artifacts.zip        ← final archive
```

## 0 — Environment & paths

In [ ]:
import os
import sys
from pathlib import Path

# ------------------------------------------------------------------ #
# Detect whether we are running on Kaggle or locally                 #
# ------------------------------------------------------------------ #
ON_KAGGLE = Path('/kaggle/input').exists()

if ON_KAGGLE:
    # Adjust these paths if your Kaggle dataset slugs differ
    REPO_ROOT   = Path('/kaggle/input/audiosepablationstudy')
    DATA_ROOT   = Path('/kaggle/input/musdb18hq')
    OUTPUT_ROOT = Path('/kaggle/working')
else:
    # Local fallback — assumes notebook is run from the repo root
    REPO_ROOT   = Path('..').resolve()
    DATA_ROOT   = REPO_ROOT / 'data' / 'musdb18hq'
    OUTPUT_ROOT = REPO_ROOT

CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
RESULTS_DIR     = OUTPUT_ROOT / 'results'
CONFIGS_DIR     = REPO_ROOT / 'configs'
SRC_DIR         = REPO_ROOT / 'src'

# Make sure src/ is importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'ON_KAGGLE   : {ON_KAGGLE}')
print(f'REPO_ROOT   : {REPO_ROOT}')
print(f'DATA_ROOT   : {DATA_ROOT}')
print(f'OUTPUT_ROOT : {OUTPUT_ROOT}')
print(f'CONFIGS_DIR : {CONFIGS_DIR}')

## 1 — Install dependencies

In [ ]:
# Install from the repo's requirements.txt
req_file = REPO_ROOT / 'requirements.txt'
!pip install -q -r {req_file}

In [ ]:
# Quick sanity checks
import torch
import torchaudio
import museval

print(f'torch      {torch.__version__}')
print(f'torchaudio {torchaudio.__version__}')
print(f'CUDA       {torch.cuda.is_available()} — device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')
print(f'museval    {museval.__version__}')

## 2 — Patch configs with Kaggle paths

We override `data.root`, `training.checkpoint_dir`, and `experiment.output_dir`
programmatically so that the original YAML files stay unchanged.

In [ ]:
import yaml

EXPERIMENTS = ['baseline', 'dsc5', 'dsc10', 'dsc20', 'dsc_full']

def load_and_patch_config(exp_name: str) -> dict:
    """Load YAML config and inject runtime paths."""
    cfg_path = CONFIGS_DIR / f'{exp_name}.yaml'
    with open(cfg_path) as fh:
        cfg = yaml.safe_load(fh)

    cfg['data']['root']                  = str(DATA_ROOT)
    cfg['training']['checkpoint_dir']    = str(CHECKPOINTS_DIR / exp_name)
    cfg['experiment']['output_dir']      = str(RESULTS_DIR)
    cfg['evaluation']['save_csv']        = str(RESULTS_DIR / f'{exp_name}_metrics.csv')
    return cfg

# Preview patched baseline config
import json
print(json.dumps(load_and_patch_config('baseline'), indent=2))

## 3 — Save patched configs to /kaggle/working

In [ ]:
PATCHED_CONFIGS_DIR = OUTPUT_ROOT / 'configs_patched'
PATCHED_CONFIGS_DIR.mkdir(exist_ok=True)

patched_paths = {}
for exp_name in EXPERIMENTS:
    cfg = load_and_patch_config(exp_name)
    out = PATCHED_CONFIGS_DIR / f'{exp_name}.yaml'
    with open(out, 'w') as fh:
        yaml.dump(cfg, fh, default_flow_style=False, allow_unicode=True)
    patched_paths[exp_name] = out
    print(f'Saved patched config: {out}')

## 4 — Training

Each cell trains one experiment. Feel free to comment out experiments you have
already completed or that you want to skip.

In [ ]:
import subprocess
import sys

SEP = '=' * 60

def run_train(exp_name: str, resume: bool = False) -> None:
    """Launch train.py for a given experiment and stream stdout."""
    cfg = patched_paths[exp_name]
    cmd = [sys.executable, str(SRC_DIR / 'train.py'), '--config', str(cfg)]
    if resume:
        cmd.append('--resume')
    print(f'\n{SEP}')
    print(f'  Training: {exp_name}')
    print(SEP)
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        print(f'⚠  Training {exp_name} exited with code {result.returncode}')
    else:
        print(f'✓  Training {exp_name} complete')

In [ ]:
run_train('baseline')

In [ ]:
run_train('dsc5')

In [ ]:
run_train('dsc10')

In [ ]:
run_train('dsc20')

In [ ]:
run_train('dsc_full')

## 5 — Evaluation

In [ ]:
def run_evaluate(exp_name: str) -> None:
    """Launch evaluate.py for a given experiment and stream stdout."""
    cfg  = patched_paths[exp_name]
    ckpt = CHECKPOINTS_DIR / exp_name / 'best.pt'
    if not ckpt.exists():
        print(f'⚠  Checkpoint not found for {exp_name}: {ckpt}  — skipping evaluation.')
        return
    cmd = [
        sys.executable, str(SRC_DIR / 'evaluate.py'),
        '--config',     str(cfg),
        '--checkpoint', str(ckpt),
    ]
    print(f'\n{SEP}')
    print(f'  Evaluating: {exp_name}')
    print(SEP)
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        print(f'⚠  Evaluation {exp_name} exited with code {result.returncode}')
    else:
        print(f'✓  Evaluation {exp_name} complete')

In [ ]:
for exp in EXPERIMENTS:
    run_evaluate(exp)

## 6 — Aggregate results table

In [ ]:
import pandas as pd
import glob

# Collect all *_metrics.csv (aggregate, not per-track)
metric_files = sorted(RESULTS_DIR.glob('*_metrics.csv'))
# Exclude per-track files
metric_files = [f for f in metric_files if 'per_track' not in f.name]

if metric_files:
    dfs = [pd.read_csv(f) for f in metric_files]
    results_df = pd.concat(dfs, ignore_index=True)
    display_cols = ['experiment', 'si_sdr', 'sdr', 'sir', 'sar', 'params', 'inference_time_ms']
    available = [c for c in display_cols if c in results_df.columns]
    print(results_df[available].to_string(index=False, float_format='{:.2f}'.format))

    # Save combined table
    combined_csv = RESULTS_DIR / 'all_experiments.csv'
    results_df.to_csv(combined_csv, index=False)
    print(f'\nCombined table saved: {combined_csv}')
else:
    print('No metrics CSV files found yet. Run evaluation cells first.')

## 7 — Archive results & checkpoints for download

In [ ]:
import zipfile
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_path = OUTPUT_ROOT / f'artifacts_{ts}.zip'

with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    # Results CSVs
    for csv_file in RESULTS_DIR.glob('*.csv'):
        zf.write(csv_file, arcname=f'results/{csv_file.name}')
    # Training logs
    for log_file in RESULTS_DIR.glob('*_train_log.csv'):
        zf.write(log_file, arcname=f'results/{log_file.name}')
    # Best checkpoints (lightweight — only best.pt per experiment)
    for exp in EXPERIMENTS:
        best_ckpt = CHECKPOINTS_DIR / exp / 'best.pt'
        if best_ckpt.exists():
            zf.write(best_ckpt, arcname=f'checkpoints/{exp}/best.pt')
            print(f'  Added checkpoint: {best_ckpt}')
        # Also include the config saved alongside the checkpoint
        cfg_backup = CHECKPOINTS_DIR / exp / 'config.yaml'
        if cfg_backup.exists():
            zf.write(cfg_backup, arcname=f'checkpoints/{exp}/config.yaml')

print(f'\nArchive saved: {archive_path}  ({archive_path.stat().st_size / 1e6:.1f} MB)')

## 8 — Quick training-curve plot (optional)

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for exp in EXPERIMENTS:
        log_csv = RESULTS_DIR / f'{exp}_train_log.csv'
        if not log_csv.exists():
            continue
        df = pd.read_csv(log_csv)
        axes[0].plot(df['epoch'], df['val_loss'],   label=exp)
        axes[1].plot(df['epoch'], df['val_si_sdr'], label=exp)

    axes[0].set_title('Validation Loss (neg SI-SDR)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].set_title('Validation SI-SDR (dB)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('SI-SDR (dB)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plot_path = RESULTS_DIR / 'training_curves.png'
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'Training curves saved: {plot_path}')

except ImportError:
    print('matplotlib not available — skipping plot.')